# BFR-CAE 推論・評価デモ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koki01150124/bfr-cae/blob/main/visualize_result.ipynb)

学習済みの CAE と BFR-CAE を読み込み，Kodak画像に対する推論，Bit Flip，復元比較，PSNR / SSIM / MS-SSIM を確認します．

本Notebookは学習を行いません．

## 1．環境初期化（Colab / ローカル）

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/koki01150124/bfr-cae.git"
REPO_NAME = "bfr-cae"

if IN_COLAB:
    os.chdir("/content")
    repo_dir = Path("/content") / REPO_NAME
    if not (repo_dir / "src").is_dir():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(repo_dir)
else:
    # ローカルでは，Notebook のあるリポジトリルートをそのまま使用する
    notebook_dir = Path.cwd()
    if not (notebook_dir / "src").is_dir():
        raise RuntimeError(
            "リポジトリルートで Notebook を開いてください．"
            f"現在のディレクトリ: {notebook_dir}"
        )

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")

In [ ]:
import importlib.util


def _missing(packages):
    return [name for name in packages if importlib.util.find_spec(name) is None]


# Google Colab の既存 PyTorch は再インストールしない
required = ["pytorch_msssim", "PIL", "matplotlib"]
if not IN_COLAB:
    required = ["torch", "torchvision"] + required

missing = _missing(required)
if missing:
    # import 名と pip 名の対応
    pip_names = {
        "pytorch_msssim": "pytorch-msssim==1.0.0",
        "PIL": "Pillow==12.3.0",
        "matplotlib": "matplotlib==3.11.2",
        "torch": "torch==2.14.0",
        "torchvision": "torchvision==0.29.0",
    }
    to_install = [pip_names[name] for name in missing]
    print("Installing:", to_install)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)
else:
    print("必要なパッケージは揃っています．")

if IN_COLAB and importlib.util.find_spec("torch") is None:
    raise RuntimeError("Colab に PyTorch が見つかりません．ランタイムを確認してください．")

## 2．設定

`IMAGE_INDEX` は Kodak の `kodim01.png`〜`kodim24.png` に対応する 1〜24 の整数です．  
`TEST_BER` は推論時のビット反転確率です．

In [ ]:
IMAGE_INDEX = 1
TEST_BER = 0.01

LATENT_CHANNELS = 128  # bpp = 128 / 64 = 2.0
PATCH_SIZE = 128
SEED = 0

CAE_CKPT = REPO_ROOT / "checkpoints" / "cae_bpp2.0_trainber0.10.pth"
BFR_CAE_CKPT = REPO_ROOT / "checkpoints" / "bfr_cae_bpp2.0_trainber0.10.pth"

print(f"IMAGE_INDEX={IMAGE_INDEX}")
print(f"TEST_BER={TEST_BER}")
print(f"LATENT_CHANNELS={LATENT_CHANNELS}, bpp={LATENT_CHANNELS / 64}")

## 3．Device / モデル読み込み

In [ ]:
import torch
import matplotlib.pyplot as plt

from src.bitflip import BitFlipChannel
from src.metrics import compute_bpp_from_latent, compute_quality_metrics
from src.utils import (
    encode_quantize,
    get_device,
    load_kodak_image,
    load_model,
    reconstruct_with_patches,
    reconstruct_with_patches_bitflip,
    set_seed,
    to_numpy_image,
)

set_seed(SEED)
device = get_device()
print(f"device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

cae = load_model(CAE_CKPT, latent_channels=LATENT_CHANNELS, device=device)
bfr_cae = load_model(BFR_CAE_CKPT, latent_channels=LATENT_CHANNELS, device=device)
print(f"loaded CAE: {CAE_CKPT.name}")
print(f"loaded BFR-CAE: {BFR_CAE_CKPT.name}")

## 4．Kodak画像の読み込み

In [ ]:
x, image_path = load_kodak_image(IMAGE_INDEX, data_dir=REPO_ROOT / "data", device=device)
print(f"file={image_path.name}, shape={tuple(x.shape)}")

plt.figure(figsize=(5, 5))
plt.imshow(to_numpy_image(x))
plt.title(image_path.name)
plt.axis("off")
plt.show()

## 5．Encoder → Binary Quantization → Bit Flip → Decoder

処理の流れを BFR-CAE で確認します．Kodak画像はパッチ単位（128×128）で推論します．

In [ ]:
z_hat = encode_quantize(bfr_cae, x, patch_size=PATCH_SIZE)
bpp = compute_bpp_from_latent(z_hat, x.shape)
print(f"z_hat shape={tuple(z_hat.shape)}, unique={torch.unique(z_hat).tolist()}")
print(f"bpp={bpp:.4f}")

channel = BitFlipChannel(TEST_BER).to(device)
z_tilde = channel(z_hat)
n_flip = int((z_hat != z_tilde).sum().item())
n_bits = z_hat.numel()
print(f"Bit Flip: TEST_BER={TEST_BER}, flipped={n_flip}/{n_bits} ({n_flip / n_bits:.4f})")

# パッチ単位の復元（研究用コードと同じ手順）
x_hat_clean = reconstruct_with_patches(bfr_cae, x, patch_size=PATCH_SIZE)
x_hat_flip = reconstruct_with_patches_bitflip(
    bfr_cae, x, p=TEST_BER, patch_size=PATCH_SIZE
)
print(f"recon clean range=[{x_hat_clean.min().item():.4f}, {x_hat_clean.max().item():.4f}]")
print(f"recon flip  range=[{x_hat_flip.min().item():.4f}, {x_hat_flip.max().item():.4f}]")

## 6．CAE / BFR-CAE の復元画像比較

In [ ]:
models = {
    "CAE": cae,
    "BFR-CAE": bfr_cae,
}

recons = {}
for name, model in models.items():
    recons[(name, 0.0)] = reconstruct_with_patches(model, x, patch_size=PATCH_SIZE)
    recons[(name, TEST_BER)] = reconstruct_with_patches_bitflip(
        model, x, p=TEST_BER, patch_size=PATCH_SIZE
    )

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
titles = [
    ["Original", "CAE (BER=0)", f"CAE (BER={TEST_BER})"],
    ["Original", "BFR-CAE (BER=0)", f"BFR-CAE (BER={TEST_BER})"],
]
images = [
    [x, recons[("CAE", 0.0)], recons[("CAE", TEST_BER)]],
    [x, recons[("BFR-CAE", 0.0)], recons[("BFR-CAE", TEST_BER)]],
]

for row in range(2):
    for col in range(3):
        axes[row, col].imshow(to_numpy_image(images[row][col]))
        axes[row, col].set_title(titles[row][col])
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 7．PSNR，SSIM，MS-SSIM

In [ ]:
print(f"IMAGE_INDEX={IMAGE_INDEX} ({image_path.name}), TEST_BER={TEST_BER}\n")

rows = []
for name in ("CAE", "BFR-CAE"):
    for ber in (0.0, TEST_BER):
        metrics = compute_quality_metrics(x, recons[(name, ber)])
        rows.append((name, ber, metrics))
        print(
            f"{name:8s}  BER={ber:<5}  "
            f"PSNR={metrics['psnr']:.4f}  "
            f"SSIM={metrics['ssim']:.6f}  "
            f"MS-SSIM={metrics['ms_ssim']:.6f}"
        )

## 補足

- モデル構造，Binary Quantization，Bit Flip，評価指標の実装は `src/` にあり，本Notebookでは再実装していません．
- checkpoint は bpp=2.0，Train BER=0.10 の CAE / BFR-CAE です．条件の詳細は README を参照してください．
- `IMAGE_INDEX` や `TEST_BER` を変更して再実行すると，別画像・別 BER での結果を確認できます．